## Tokenizer for parsing the text

In [19]:
import re
from collections import defaultdict

def parse_multi_report_pubtator(raw_data):
    # If the input is bytes, decode it to a string first
    if isinstance(raw_data, bytes):
        raw_data = raw_data.decode('utf-8')

    report_groups = defaultdict(lambda: {'title': '', 'abstract': '', 'annotations': []})
    lines = raw_data.strip().split('\n')
    
    # 1. Grouping Phase: Separate data by PMID
    for line in lines:
        if not line.strip(): 
            continue
        
        parts = line.split('|')
        if len(parts) > 2:
            pmid, type_flag, content = parts[0], parts[1], parts[2]
            if type_flag == 't':
                report_groups[pmid]['title'] = content
            elif type_flag == 'a':
                report_groups[pmid]['abstract'] = content
        else:
            tab_parts = line.split('\t')
            if len(tab_parts) >= 5:
                pmid = tab_parts[0]
                report_groups[pmid]['annotations'].append({
                    'start': int(tab_parts[1]),
                    'end': int(tab_parts[2]),
                    'tags': tab_parts[4].split(',')
                })

    all_results = []

    # 2. Processing Phase: Handle each report independently
    for pmid, data in report_groups.items():
        # Using a space to separate title and abstract to maintain offset integrity
        full_text = data['title'] + " " + data['abstract']
        
        report_words = []
        report_tags = []
        
        for match in re.finditer(r'\w+', full_text):
            start, end = match.start(), match.end()
            word = match.group()
            
            current_word_tags = set()
            for ann in data['annotations']:
                if start >= ann['start'] and end <= ann['end']:
                    for t in ann['tags']:
                        current_word_tags.add(t)
            
            report_words.append(word)
            
            # --- Logic Update Start ---
            # If the set is empty, we tag it with 'O'
            if not current_word_tags:
                report_tags.append(['O'])
            else:
                report_tags.append(sorted(list(current_word_tags)))
            # --- Logic Update End ---
        
        all_results.append({
            'pmid': pmid,
            'words': report_words,
            'tags': report_tags
        })
        
    return all_results

In [ ]:
# Loading the full text
with open("corpus_pubtator.txt", "r") as f:
    full_text = f.read()

In [ ]:
# Retrieving results
results = parse_multi_report_pubtator2(full_text)

In [ ]:
# Checking if it is done correctly
results[0]["tags"]

[['T116', 'T123'],
 ['O'],
 ['O'],
 ['O'],
 ['O'],
 ['T047'],
 ['T047'],
 ['T047'],
 ['T047'],
 ['O'],
 ['T047'],
 ['T047'],
 ['T047'],
 ['T047'],
 ['T047'],
 ['T047'],
 ['O'],
 ['T047'],
 ['T047'],
 ['T047'],
 ['T101'],
 ['O'],
 ['O'],
 ['O'],
 ['O'],
 ['T079'],
 ['T079'],
 ['T047'],
 ['T047'],
 ['O'],
 ['T169'],
 ['T169'],
 ['O'],
 ['T047'],
 ['T047'],
 ['T047'],
 ['T047'],
 ['O'],
 ['O'],
 ['O'],
 ['T033'],
 ['T033'],
 ['T033'],
 ['T033'],
 ['T033'],
 ['T033'],
 ['T033'],
 ['T033'],
 ['O'],
 ['T081'],
 ['O'],
 ['T033'],
 ['O'],
 ['T169'],
 ['T169'],
 ['O'],
 ['O'],
 ['T063'],
 ['T063'],
 ['O'],
 ['T052'],
 ['T052'],
 ['T052'],
 ['O'],
 ['O'],
 ['O'],
 ['O'],
 ['O'],
 ['T116'],
 ['O'],
 ['T116', 'T123'],
 ['T116', 'T123'],
 ['T116', 'T123'],
 ['O'],
 ['O'],
 ['T047'],
 ['T047'],
 ['O'],
 ['T047'],
 ['O'],
 ['O'],
 ['O'],
 ['T047'],
 ['T047'],
 ['O'],
 ['O'],
 ['O'],
 ['O'],
 ['T062'],
 ['O'],
 ['O'],
 ['T169'],
 ['O'],
 ['O'],
 ['O'],
 ['T116', 'T123'],
 ['T033'],
 ['T116'],
 ['O'],
